In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

# TASK 1: Indexing & Selection (Pembersihan Anomali & Imputasi)
df['harga_satuan'] = df['harga_satuan'].astype(str).str.replace(',', '.').astype(float)
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'
kolom_target = ['id_transaksi', 'tanggal', 'kode_gerai', 'jumlah']
indeks_kolom = df.columns.get_indexer(kolom_target)
subset_iloc = df.iloc[:5, indeks_kolom]
print("--- Task 1.4: Hasil Slicing .iloc[] (5 Baris Pertama) ---")
display(subset_iloc)

# TASK 2: Operasi Aritmatika (Feature Engineering)
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100.0)
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11

# TASK 3: Filtering (Penyaringan Data)
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]
print(f"\n--- Task 3.1 ---")
print(f"Jumlah baris data yang memenuhi kriteria multi-kondisi: {len(filter_multi)}")
filter_query = df.query('kode_gerai == "G10" and jumlah >= 5')
print("\n--- Task 3.2: Hasil Filter Query Gerai G10 ---")
display(filter_query)

# TASK 4: Sorting (Pengurutan Data)
df_sorted = df.sort_values(by=['kode_gerai', 'Total_Bersih'], ascending=[True, False])
top_5_transaksi = df.nlargest(5, 'Total_Bersih')
print("\n--- Task 4.2: Top 5 Transaksi Berdasarkan Total Bersih Tertinggi ---")
display(top_5_transaksi)

# TASK 5: Grouping & Agregasi (Pelaporan Kinerja Gerai)
df_selesai = df[df['status'] == 'SELESAI']
laporan_gerai = df_selesai.groupby('kode_gerai', as_index=False).agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(by='Total_Omzet_Bersih', ascending=False).round(2)

print("\n--- Task 5.1: Laporan Agregasi Per Gerai ---")
display(laporan_gerai)
total_per_gerai = df.groupby('kode_gerai')['Total_Bersih'].transform('sum')
df['Kontribusi_Gerai_Persen'] = (df['Total_Bersih'] / total_per_gerai) * 100
print("\n--- Task 5.2: Sampel Data dengan Kolom Kontribusi Gerai (%) ---")
display(df[['id_transaksi', 'kode_gerai', 'Total_Bersih', 'Kontribusi_Gerai_Persen']].head())

--- Task 1.4: Hasil Slicing .iloc[] (5 Baris Pertama) ---


,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1



--- Task 3.1 ---
Jumlah baris data yang memenuhi kriteria multi-kondisi: 139

--- Task 3.2: Hasil Filter Query Gerai G10 ---


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,52000.0,10,SELESAI,312000.0,31200.0,311688.0
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,18000.0,0,SELESAI,108000.0,0.0,119880.0
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,6000.0,0,SELESAI,36000.0,0.0,39960.0
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,54000.0,0,SELESAI,324000.0,0.0,359640.0
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,29000.0,0,SELESAI,174000.0,0.0,193140.0



--- Task 4.2: Top 5 Transaksi Berdasarkan Total Bersih Tertinggi ---


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
994,TRX0241002,2024-03-18,G11,K038,MKN1101,2093,48000.0,10,SELESAI,100464000.0,10046400.0,100363536.0
4985,TRX0244221,2024-12-09,G09,K005,BYI1500,1301,51500.0,10,SELESAI,67001500.0,6700150.0,66934498.5
467,TRX0243696,2024-02-27,G06,K_UNKNOWN,SGR1201,892,54000.0,0,SELESAI,48168000.0,0.0,53466480.0
3393,TRX0241029,2025-05-12,G07,K028,SGR1205,1983,23000.0,10,SELESAI,45609000.0,4560900.0,45563391.0
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0



--- Task 5.1: Laporan Agregasi Per Gerai ---


,kode_gerai,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
10,G11,398,3170,1.253536e+08,314958.82
5,G06,383,3103,9.113936e+07,237961.77
8,G09,379,3027,9.019606e+07,237984.32
6,G07,384,4284,8.036181e+07,209275.55
0,G01,450,2635,7.077959e+07,157287.97
2,G03,372,3274,5.980970e+07,160778.77
9,G10,401,1742,5.892248e+07,146938.85
7,G08,408,1700,3.550256e+07,87016.08
11,G12,406,1010,2.464483e+07,60701.55
3,G04,399,1042,2.453577e+07,61493.15



--- Task 5.2: Sampel Data dengan Kolom Kontribusi Gerai (%) ---


,id_transaksi,kode_gerai,Total_Bersih,Kontribusi_Gerai_Persen
0,TRX0241065,G09,57720.0,0.063031
1,TRX0240510,G09,23310.0,0.025455
2,TRX0244483,G11,24975.0,0.018364
3,TRX0242532,G05,14430.0,0.060124
4,TRX0243509,G05,6660.0,0.027750
